In [1]:
import pandas as pd
import re
from pathlib import Path

In [2]:
# Set paths
project_root = Path(r"D:\Extract Data\Potato_Pathogen_Project")
taxonomy_path = project_root / "data/reference/UNITE_ITS/sh_qiime_release_04.04.2024/developer/exported-taxonomy/taxonomy.tsv"
otu_path = project_root / "data/reference/UNITE_ITS/sh_qiime_release_04.04.2024/developer/phyloseq/otu_table.tsv"
metadata_path = project_root / "data/metadata/metadata.csv"

print(f"Taxonomy exists: {taxonomy_path.exists()}")
print(f"OTU table exists: {otu_path.exists()}")
print(f"Metadata exists: {metadata_path.exists()}")

Taxonomy exists: True
OTU table exists: True
Metadata exists: True


In [3]:
# Load taxonomy.tsv
taxonomy_df = pd.read_csv(taxonomy_path, sep='\t')
print(f"Shape: {taxonomy_df.shape}")
print(f"Columns: {list(taxonomy_df.columns)}")
taxonomy_df.head()

Shape: (3325, 3)
Columns: ['Feature ID', 'Taxon', 'Confidence']


,Feature ID,Taxon,Confidence
0,2f1a972947e062cd9bcc624150e4ba33,k__Fungi;p__Ascomycota;c__Sordariomycetes;o__S...,-1.0
1,e224684a99a6a3fb44d768fab2b76d36,k__Fungi;p__Ascomycota;c__Sordariomycetes;o__S...,-1.0
2,214bda0560c6f3bc31a21b896617dbf2,k__Fungi;p__Ascomycota;c__Sordariomycetes;o__S...,-1.0
3,07a13ac8811ce0d88620ebef87a7533f,k__Fungi;p__Ascomycota;c__Lecanoromycetes;o__L...,-1.0
4,f0edca8501984df86db2e17940935f1b,k__Fungi;p__Ascomycota;c__Lecanoromycetes;o__L...,-1.0


In [4]:
# Load otu_table.tsv
otu_df = pd.read_csv(otu_path, sep='\t', index_col=0)
print(f"Shape: {otu_df.shape[0]} features × {otu_df.shape[1]} samples")
print(f"Sample columns: {list(otu_df.columns[:5])}...")
otu_df.iloc[:5, :5]

Shape: 3325 features × 17 samples
Sample columns: ['DNA_1', 'DNA_10', 'DNA_11', 'DNA_12', 'DNA_13']...


,DNA_1,DNA_10,DNA_11,DNA_12,DNA_13
2f1a972947e062cd9bcc624150e4ba33,0.0,0.0,0.0,51.0,0.0
e224684a99a6a3fb44d768fab2b76d36,0.0,0.0,0.0,0.0,0.0
214bda0560c6f3bc31a21b896617dbf2,0.0,0.0,0.0,0.0,0.0
07a13ac8811ce0d88620ebef87a7533f,0.0,0.0,0.0,0.0,0.0
f0edca8501984df86db2e17940935f1b,0.0,0.0,0.0,0.0,0.0


In [7]:
# Extract unique genera from taxonomy
def extract_genus(taxon_string):
    """Extract genus from UNITE taxonomy string"""
    if pd.isna(taxon_string):
        return "Unassigned"
    match = re.search(r'g__([^;]+)', taxon_string)
    if match:
        genus = match.group(1).strip()
        return genus if genus and genus != "unidentified" else "Unidentified"
    return "Unassigned"

taxonomy_df['Genus'] = taxonomy_df['Taxon'].apply(extract_genus)
unique_genera = taxonomy_df[taxonomy_df['Genus'] != "Unassigned"]['Genus'].unique()
print(f"Total unique genera: {len(unique_genera)}")
print(f"\nFirst 30 genera:")
for g in sorted(unique_genera)[:30]:
    print(f"  {g}")

Total unique genera: 19

First 30 genera:
  Acarosporaceae_gen_Incertae_sedis
  Alternaria
  Catenaria
  Chaetosphaeria
  Fusarium
  GS01_phy_Incertae_sedis_gen_Incertae_sedis
  GS04_gen_Incertae_sedis
  Kurokawia
  Metschnikowia
  Polyschema
  Ramalina
  Rhizophydium
  Rhizoplaca
  Sarocladium
  Sordariales_gen_Incertae_sedis
  Spizellomycetaceae_gen_Incertae_sedis
  Tomentella
  Trichoderma
  Tulasnellaceae_gen_Incertae_sedis


In [8]:
# Genus frequency count
genus_counts = taxonomy_df['Genus'].value_counts()
print("Top 30 most frequent genera:")
for genus, count in genus_counts.head(30).items():
    print(f"  {genus}: {count} features")

Top 30 most frequent genera:
  Sordariales_gen_Incertae_sedis: 1279 features
  Rhizoplaca: 1084 features
  Polyschema: 285 features
  Metschnikowia: 234 features
  Chaetosphaeria: 221 features
  Kurokawia: 86 features
  Tulasnellaceae_gen_Incertae_sedis: 35 features
  Fusarium: 31 features
  Ramalina: 18 features
  Tomentella: 14 features
  GS01_phy_Incertae_sedis_gen_Incertae_sedis: 13 features
  Alternaria: 13 features
  Acarosporaceae_gen_Incertae_sedis: 3 features
  GS04_gen_Incertae_sedis: 2 features
  Rhizophydium: 2 features
  Sarocladium: 2 features
  Catenaria: 1 features
  Spizellomycetaceae_gen_Incertae_sedis: 1 features
  Trichoderma: 1 features


In [9]:
# Known potato fungal pathogens (for matching)
known_pathogens = {
    "Alternaria": ["Alternaria solani", "Alternaria alternata"],
    "Fusarium": ["Fusarium oxysporum", "Fusarium solani", "Fusarium sambucinum"],
    "Verticillium": ["Verticillium dahliae", "Verticillium albo-atrum"],
    "Rhizoctonia": ["Rhizoctonia solani"],
    "Colletotrichum": ["Colletotrichum coccodes", "Colletotrichum acutatum"],
    "Phytophthora": ["Phytophthora infestans", "Phytophthora erythroseptica"],
    "Helminthosporium": ["Helminthosporium solani"],
    "Polyscytalum": ["Polyscytalum pustulans"],
    "Spongospora": ["Spongospora subterranea"],
    "Pythium": ["Pythium ultimum", "Pythium aphanidermatum"],
    "Sclerotinia": ["Sclerotinia sclerotiorum"],
    "Macrophomina": ["Macrophomina phaseolina"],
    "Phoma": ["Phoma exigua"],
    "Boeremia": ["Boeremia exigua"]
}

print(f"Known pathogen genera to check: {len(known_pathogens)}")
print(list(known_pathogens.keys()))

Known pathogen genera to check: 14
['Alternaria', 'Fusarium', 'Verticillium', 'Rhizoctonia', 'Colletotrichum', 'Phytophthora', 'Helminthosporium', 'Polyscytalum', 'Spongospora', 'Pythium', 'Sclerotinia', 'Macrophomina', 'Phoma', 'Boeremia']


In [10]:
# Check which known pathogen genera are present
pathogen_matches = []
for genus, species_list in known_pathogens.items():
    genus_rows = taxonomy_df[taxonomy_df['Genus'] == genus]
    if len(genus_rows) > 0:
        print(f"✓ {genus}: {len(genus_rows)} features found")
        for _, row in genus_rows.iterrows():
            pathogen_matches.append({
                'Genus': genus,
                'Feature_ID': row['Feature ID'],
                'Taxon': row['Taxon'],
                'Confidence': row['Confidence']
            })
    else:
        print(f"✗ {genus}: NOT DETECTED")

print(f"\nTotal pathogen genus matches: {len(pathogen_matches)}")

✓ Alternaria: 13 features found
✓ Fusarium: 31 features found
✗ Verticillium: NOT DETECTED
✗ Rhizoctonia: NOT DETECTED
✗ Colletotrichum: NOT DETECTED
✗ Phytophthora: NOT DETECTED
✗ Helminthosporium: NOT DETECTED
✗ Polyscytalum: NOT DETECTED
✗ Spongospora: NOT DETECTED
✗ Pythium: NOT DETECTED
✗ Sclerotinia: NOT DETECTED
✗ Macrophomina: NOT DETECTED
✗ Phoma: NOT DETECTED
✗ Boeremia: NOT DETECTED

Total pathogen genus matches: 44


In [11]:
# Calculate prevalence for detected pathogen genera
print("Calculating sample prevalence...\n")

# Get actual sample columns (excluding metadata columns if any)
sample_cols = [col for col in otu_df.columns if col.startswith('DNA_')]
print(f"Sample columns found: {len(sample_cols)}")

for genus in known_pathogens.keys():
    genus_rows = taxonomy_df[taxonomy_df['Genus'] == genus]
    if len(genus_rows) > 0:
        feature_ids = genus_rows['Feature ID'].tolist()
        samples_positive = set()
        total_reads = 0
        for fid in feature_ids:
            if fid in otu_df.index:
                sample_counts = otu_df.loc[fid, sample_cols]
                positive_samples = sample_counts[sample_counts > 0].index.tolist()
                samples_positive.update(positive_samples)
                total_reads += sample_counts.sum()
        prevalence = len(samples_positive) / len(sample_cols) * 100
        print(f"{genus}: {prevalence:.1f}% prevalence ({len(samples_positive)}/{len(sample_cols)} samples), {total_reads:,.0f} total reads")

Calculating sample prevalence...

Sample columns found: 17
Alternaria: 100.0% prevalence (17/17 samples), 171,416 total reads
Fusarium: 100.0% prevalence (17/17 samples), 270,216 total reads


In [12]:
# Build complete summary
summary_data = []
for genus in unique_genera:
    genus_rows = taxonomy_df[taxonomy_df['Genus'] == genus]
    feature_ids = genus_rows['Feature ID'].tolist()
    
    samples_positive = set()
    total_reads = 0
    for fid in feature_ids:
        if fid in otu_df.index:
            sample_counts = otu_df.loc[fid, sample_cols]
            samples_positive.update(sample_counts[sample_counts > 0].index.tolist())
            total_reads += sample_counts.sum()
    
    prevalence = len(samples_positive) / len(sample_cols) * 100 if len(sample_cols) > 0 else 0
    is_pathogen = genus in known_pathogens.keys()
    
    summary_data.append({
        'Genus': genus,
        'Is_Known_Pathogen': is_pathogen,
        'Prevalence_%': round(prevalence, 1),
        'Samples_Positive': len(samples_positive),
        'Total_Reads': int(total_reads),
        'Feature_Count': len(feature_ids)
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Total_Reads', ascending=False)

print(f"Total genera: {len(summary_df)}")
print(f"Known pathogen genera present: {len(summary_df[summary_df['Is_Known_Pathogen'] == True])}")
print(f"\nTop 20 most abundant genera:")
summary_df.head(20)

Total genera: 19
Known pathogen genera present: 2

Top 20 most abundant genera:


,Genus,Is_Known_Pathogen,Prevalence_%,Samples_Positive,Total_Reads,Feature_Count
1,Rhizoplaca,False,100.0,17,348747,1084
14,Polyschema,False,100.0,17,331167,285
17,Fusarium,True,100.0,17,270216,31
0,Sordariales_gen_Incertae_sedis,False,100.0,17,258119,1279
11,Chaetosphaeria,False,100.0,17,239658,221
13,Alternaria,True,100.0,17,171416,13
2,Metschnikowia,False,100.0,17,57930,234
5,Kurokawia,False,100.0,17,11605,86
4,Ramalina,False,76.5,13,2816,18
6,Tulasnellaceae_gen_Incertae_sedis,False,94.1,16,1298,35


In [13]:
# Create results directory
results_dir = project_root / "results/taxonomy"
results_dir.mkdir(parents=True, exist_ok=True)

# Save summary
summary_df.to_csv(results_dir / "fungal_genera_summary.csv", index=False)
print(f"✓ Saved: {results_dir / 'fungal_genera_summary.csv'}")

# Save pathogen matches
if pathogen_matches:
    pd.DataFrame(pathogen_matches).to_csv(results_dir / "pathogen_matches.csv", index=False)
    print(f"✓ Saved: {results_dir / 'pathogen_matches.csv'}")

print("\n✅ Analysis complete!")

✓ Saved: D:\Extract Data\Potato_Pathogen_Project\results\taxonomy\fungal_genera_summary.csv
✓ Saved: D:\Extract Data\Potato_Pathogen_Project\results\taxonomy\pathogen_matches.csv

✅ Analysis complete!
